# 07 · Grounded NCI PDQ guidance assistant — M3

**Project:** Skin Lesion Triage · BIU DS23 Capstone
**Owner:** Agent hat · **Charter:** M3 (groundedness) and the agent's retrieval tool

## What this notebook does
1. Downloads 7 NCI PDQ Health Professional summaries on skin cancer and saves them as a frozen corpus.
2. Splits them into chunks with stable ids (`pdq:<doc>:sNN:cNN`), embeds them and builds a Chroma index.
3. Answers questions **only** from retrieved chunks, citing their ids, and says "I don't know" otherwise.
4. Measures M3 on 20 frozen questions, before and after cross-encoder re-ranking.
5. Independently tests refusal behavior after re-ranking and consolidates three reported evaluation runs, distinguishing historical results from the current verification.


> **Why the agent needs it.** The agent's tool `retrieve_guidance_context` reads this same index. Every citation
> in the dermatologist's report is a chunk id from here, and the guardrails check that the quote appears in that
> chunk word for word. Without the corpus the agent still runs, and the report says "No verified guidance found".

> **Runs on its own.** Needs no model and no other notebook. Needs `GEMINI_API_KEY` in Colab Secrets and the
> `agent/` folder in the project folder on Drive.

## Target numbers (from the charter)
| | |
|---|---|
| Frozen evaluation questions | 20 (15 answerable, 5 not in the corpus) |
| Refusals on the 5 out-of-corpus questions | 5 / 5 |
| **Groundedness** (cited ids that were actually retrieved) | **≥ 0.9** |
| Numbers in an answer that appear in the cited chunk | all |

## Outputs
| File | Contents |
|---|---|
| `data/processed/rag/nci_corpus.json` | the frozen NCI PDQ corpus (09_agent_daily_report and 08_agent_m4_vs_ifelse read it) |
| `data/processed/rag/chroma_nci/` | the vector index (rebuilt from the corpus, not committed to Git) |
| `reports/agent/m3_summary_colab_review.csv` | Consolidated M3 summary of three reported runs, with evidence provenance and refusal results |
| `reports/agent/m3_answers_colab_review.csv` | Per-question answers, groundedness, and number checks from the current verification run only |
| `reports/agent/m3_refusals_after_colab_review.csv` | Per-question refusal results after re-ranking in the current verification run |


In [15]:
# Cell 1 · Configuration (identical in every notebook)
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = "/content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers"
else:
    PROJECT = os.path.abspath("..")

RAW       = os.path.join(PROJECT, "data", "raw")
PROCESSED = os.path.join(PROJECT, "data", "processed")
FIGURES   = os.path.join(PROJECT, "reports", "figures")

# Where the image archive is stored. Kept as its own setting so the heavy files can later move to a
# shared folder without touching anything else. Colab addresses Drive by path, never by the id in a
# share link, so any override below must be a path under /content/drive/MyDrive/.
IMAGES_DIR = os.path.join(RAW, "images")

for d in (RAW, PROCESSED, FIGURES, IMAGES_DIR):
    os.makedirs(d, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
# Cell 2 · Agent paths and the API key
# The agent code lives in the project folder on Drive, next to the data: PROJECT/agent/*.py
AGENT_DIR = os.path.join(PROJECT, "agent")                # triage_agent.py, rag.py, m4_eval.py, make_daily_list.py
RAG_DIR   = os.path.join(PROCESSED, "rag")                # nci_corpus.json + the Chroma index
AGENT_OUT = os.path.join(PROJECT, "reports", "agent")     # Word reports, agent logs, M3 / M4 tables
for d in (RAG_DIR, AGENT_OUT):
    os.makedirs(d, exist_ok=True)

os.environ["RAG_DIR"] = RAG_DIR                           # must be set before importing rag / triage_agent
if AGENT_DIR not in sys.path:
    sys.path.insert(0, AGENT_DIR)

def get_secret(name):
    """API keys come only from Colab Secrets (key icon, left bar). A key pasted into a cell is a leaked key."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

assert os.path.exists(os.path.join(AGENT_DIR, "triage_agent.py")), \
    f"agent code not found: upload the agent/ folder to {AGENT_DIR}"
print("agent code :", AGENT_DIR)
print("outputs    :", AGENT_OUT)

agent code : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/agent
outputs    : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/reports/agent


In [17]:
# Cell 3 · Install, seeds
!pip install -q sentence-transformers chromadb beautifulsoup4 requests openai pandas
import json, time, random
import numpy as np, pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
import rag

## 1 · The corpus

7 NCI PDQ **Health Professional** summaries: screening, prevention, melanoma treatment, skin cancer treatment,
genetics, childhood melanoma, childhood skin cancer. Downloaded once and frozen in `nci_corpus.json`, so every
later run, and every teammate, retrieves from the same text (`rag.load_or_fetch_corpus(refresh=True)` downloads again).

Licence: PDQ text may be reused as text with credit to NCI; excerpts are not an official PDQ summary.

In [18]:
# Cell 4 · Download once, then read the frozen copy
corpus = rag.load_or_fetch_corpus()
for key, d in corpus.items():
    print(f"{key:15s} {len(d['sections']):3d} sections  {sum(len(s['text']) for s in d['sections']):>8,} chars  {d['title']}")

screening         9 sections    13,009 chars  Skin Cancer Screening (PDQ) - HP
prevention       31 sections    21,083 chars  Skin Cancer Prevention (PDQ) - HP
melanoma_tx      42 sections   123,426 chars  Melanoma Treatment (PDQ) - HP
skin_tx          40 sections    51,383 chars  Skin Cancer Treatment (PDQ) - HP
genetics         84 sections   213,544 chars  Genetics of Skin Cancer (PDQ) - HP
child_melanoma   10 sections    34,319 chars  Childhood Melanoma Treatment (PDQ) - HP
child_skin        8 sections     9,460 chars  Childhood BCC and SCC of the Skin Treatment (PDQ) - HP


## 2 · The LLM backend

Gemini is reached through its OpenAI-compatible endpoint.
Temperature is fixed at 0 to reduce randomness and improve comparability between the before and after evaluations. However, identical responses across runs are not guaranteed, as demonstrated by the observed variation in baseline groundedness.


In [19]:
# Cell 5 · chat(): one function for every LLM call in this notebook
BACKEND = "gemini"
MODEL = "gemini-3.5-flash-lite"           # same model as TriageLLM in Studio
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
KEY = get_secret("GEMINI_API_KEY")
MIN_GAP_S = 13                            # free tier: about 4-5 calls a minute
_last = [0.0]

def chat(messages, temperature=0.0, max_tokens=600):
    from openai import OpenAI
    client = OpenAI(api_key=KEY, base_url=BASE_URL)
    for attempt in range(4):
        wait = MIN_GAP_S - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
        try:
            # room for the model's thinking AND the answer with its citation
            r = client.chat.completions.create(model=MODEL, messages=messages,
                                               temperature=temperature, max_tokens=max(max_tokens, 4096))
            text = (r.choices[0].message.content or "").strip()
            if text:
                return text
            print("[empty reply, retrying]")
        except Exception as e:
            print(f"[retry {attempt + 1}] {type(e).__name__}")
            time.sleep(20 * (attempt + 1))
    # A failed call must STOP the run, not be counted as "I don't know"
    raise RuntimeError("LLM unavailable (quota?). Results would be invalid - try again later.")

print("warm-up:", chat([{"role": "user", "content": "Reply with one word: READY"}], max_tokens=20))

warm-up: READY


## 3 · The frozen evaluation set

Written **before** any result is seen, then locked. 15 questions answerable from the corpus (open the source and
confirm the answer is there), 5 that are not.

In [20]:
# Cell 6 · 20 questions, locked
EVAL = [
    {"q": "What are the risk factors for melanoma?",                          "in_corpus": True},
    {"q": "How accurate is the clinical diagnosis of melanoma?",              "in_corpus": True},
    {"q": "Does skin cancer screening reduce melanoma mortality?",            "in_corpus": True},
    {"q": "What are the harms of skin cancer screening?",                     "in_corpus": True},
    {"q": "Does sunscreen use prevent skin cancer?",                          "in_corpus": True},
    {"q": "How is actinic keratosis treated?",                                "in_corpus": True},
    {"q": "What are the main types of skin cancer?",                          "in_corpus": True},
    {"q": "How is localized basal cell carcinoma treated?",                   "in_corpus": True},
    {"q": "How does melanoma in children differ from melanoma in adults?",    "in_corpus": True},
    {"q": "Which inherited genes increase the risk of melanoma?",             "in_corpus": True},
    {"q": "How is squamous cell carcinoma of the skin treated?",              "in_corpus": True},
    {"q": "What is the role of sentinel lymph node biopsy in melanoma?",      "in_corpus": True},
    {"q": "How is melanoma staged?",                                          "in_corpus": True},
    {"q": "Does indoor tanning increase the risk of skin cancer?",            "in_corpus": True},
    {"q": "What treatments are used for stage IV melanoma?",                  "in_corpus": True},
    {"q": "What is the waiting time for a dermatologist at Clalit?",          "in_corpus": False},
    {"q": "Which smartphone camera takes the best photos of moles?",          "in_corpus": False},
    {"q": "How much does a skin biopsy cost in Israel?",                      "in_corpus": False},
    {"q": "What is the recommended ibuprofen dose after a biopsy?",           "in_corpus": False},
    {"q": "Which health fund app supports photo uploads?",                    "in_corpus": False},
]
assert len(EVAL) == 20, "write exactly 20 questions"
assert sum(not e["in_corpus"] for e in EVAL) == 5, "exactly 5 must be out of the corpus"
print("eval set locked:", len(EVAL))

eval set locked: 20


## 4 · The metric

**Groundedness** = the share of chunk ids cited in an answer that were actually retrieved for that question
(Module 5, with chunk ids instead of arXiv ids). An answer that cites nothing scores 0. The cell checks the
metric on three hand-made cases before it is trusted.

In [21]:
# Cell 7 · Verify the metric before using it
assert rag.groundedness("See [pdq:screening:s04:c02].", ["pdq:screening:s04:c02"]) == 1.0
assert rag.groundedness("See [pdq:screening:s04:c02] and [pdq:genetics:s01:c00].", ["pdq:screening:s04:c02"]) == 0.5
assert rag.groundedness("No citation here.", ["pdq:screening:s04:c02"]) == 0.0
print("groundedness metric verified")

groundedness metric verified


## 5 · Chunk, embed, index

PDQ sections run to thousands of characters, so a whole section would mix several ideas in one vector.
Chunks of ~900 characters with ~150 overlap, cut on sentence boundaries, keep about one idea per chunk; the
overlap keeps a sentence that falls on a boundary. Embedding: `all-MiniLM-L6-v2`, normalized.

In [22]:
# Cell 8 · Build the index (about a minute)
rag.build_index()

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

indexed 724 chunks from 7 documents


724

In [23]:
# Cell 9 · One question end to end: the answer and the ids it was allowed to cite
ans, ids = rag.answer(EVAL[0]["q"], chat, k=3)
print(ans, "\n\nretrieved:", ids)

Risk factors for melanoma include both intrinsic (genetic and phenotype) and extrinsic (environmental or exposure) factors, which include pigmentary characteristics, as well as a family and personal history of melanoma [pdq:melanoma_tx:s02:c00]. 

retrieved: ['pdq:melanoma_tx:s02:c00', 'pdq:genetics:s47:c00', 'pdq:genetics:s69:c01']


## 6 · The "I don't know" test

The 5 out-of-corpus questions must be refused. An answer here means the model used its own knowledge,
which is exactly what a grounded assistant may not do.

In [24]:
# Cell 10 · Refusals on the 5 out-of-corpus questions
out_of_scope = [e for e in EVAL if not e["in_corpus"]]
refused = 0
for e in out_of_scope:
    a, _ = rag.answer(e["q"], chat, k=3)
    refused += rag.is_refusal(a)
    if not rag.is_refusal(a):
        print("ANSWERED ANYWAY:", e["q"], "->", a[:150])
    time.sleep(4)
print(f"refused {refused} of {len(out_of_scope)}   (target 5 of 5)")
refused_after = None   # cell 12b measures this; running cell 10 invalidates any earlier value

refused 5 of 5   (target 5 of 5)


## 7 · Before: the baseline on the 15 answerable questions

In [25]:
# Cell 11 · Baseline: groundedness, numbers found in the cited chunk, wrong refusals
def evaluate(**kw):
    rows = []
    for e in EVAL:
        if not e["in_corpus"]:
            continue
        a, ids = rag.answer(e["q"], chat, k=3, **kw)
        rows.append({"q": e["q"], "groundedness": rag.groundedness(a, ids),
                     "numbers_ok": rag.numbers_supported(a), "refused": rag.is_refusal(a), "answer": a})
        time.sleep(4)
    return pd.DataFrame(rows)

before = evaluate()
print("groundedness:", round(before.groundedness.mean(), 3), "| numbers in cited chunk:",
      f"{before.numbers_ok.sum()}/{len(before)}", "| wrongly refused:", before.refused.sum())

groundedness: 0.733 | numbers in cited chunk: 15/15 | wrongly refused: 4


## 8 · One improvement: re-ranking

Retrieve wide (20 chunks), then re-rank with a cross-encoder (`ms-marco-MiniLM-L-6-v2`) and keep 3.
PDQ treatment sections are long and alike, so the embedding's top 3 often include near-misses; a cross-encoder
reads the question and the chunk together. **Only this changes**: questions, prompt, k and model stay the same.

In [26]:
# Cell 12 · After: the same evaluation with re-ranking
after = evaluate(rerank=True)
print(f"before {before.groundedness.mean():.3f} | after {after.groundedness.mean():.3f} | "
      f"lift {after.groundedness.mean() - before.groundedness.mean():+.3f}   (target >= 0.9)")

before 0.733 | after 0.933 | lift +0.200   (target >= 0.9)


In [27]:

# Cell 12b · NEW: Test refusals AFTER re-ranking

assert len(out_of_scope) == 5, "Expected 5 frozen out-of-corpus questions"

refused_after = 0
refusal_checks_after = []

for e in out_of_scope:
    answer, retrieved_ids = rag.answer(
        e["q"],
        chat,
        k=3,
        rerank=True
    )

    is_refusal = bool(rag.is_refusal(answer))
    refused_after += int(is_refusal)

    refusal_checks_after.append({
        "question": e["q"],
        "refused": is_refusal,
        "answer": answer,
        "retrieved_ids": ";".join(retrieved_ids)
    })

    print("PASS" if is_refusal else "FAIL", "|", e["q"])
    print("Answer:", answer[:300])
    print("-" * 60)

    time.sleep(4)

print(
    f"AFTER re-ranking: {refused_after}/"
    f"{len(out_of_scope)} refusals"
)

if refused_after == 5:
    print("Refusal target met in this run.")
else:
    print("Refusal target NOT met. Review failed cases.")


PASS | What is the waiting time for a dermatologist at Clalit?
Answer: I don't know
------------------------------------------------------------
PASS | Which smartphone camera takes the best photos of moles?
Answer: I don't know
------------------------------------------------------------
PASS | How much does a skin biopsy cost in Israel?
Answer: I don't know
------------------------------------------------------------
PASS | What is the recommended ibuprofen dose after a biopsy?
Answer: I don't know
------------------------------------------------------------
PASS | Which health fund app supports photo uploads?
Answer: I don't know
------------------------------------------------------------
AFTER re-ranking: 5/5 refusals
Refusal target met in this run.



**Interpretation.** In the current verification run, re-ranking improved citation-ID groundedness from 0.733 to 0.933 (+0.200), exceeding the M3 target of 0.90.

Before re-ranking, 4 of the 15 answerable questions were incorrectly refused. After re-ranking, citation-ID groundedness reached 0.933. This suggests that re-ranking helped retrieve passages needed to answer more of the questions.

All 5 out-of-corpus questions were correctly refused both before re-ranking (Cell 10) and after re-ranking (Cell 12b). The post-reranking refusal result was measured directly in this verification run.

### Run-to-run variation

| Run | Groundedness before | Wrong refusals before | Groundedness after | Refusals after |
|---|---|---|---|---|
| Early run (reported) | 0.600 | 6/15 | 0.933 | Not evaluated |
| Committed run (30.10.2026) | 0.733 | 4/15 | 0.933 | Not evaluated |
| Verification run (31.10.2026) | 0.733 | 4/15 | 0.933 | 5/5 |

Across the three reported runs, baseline groundedness ranged from 0.600 to 0.733, while groundedness after re-ranking was reported as 0.933 in each run.

The observed improvement ranged from +0.200 to +0.333 (20.0–33.3 percentage points). Thus, the improvement is not dependent on selecting a single baseline result.

The early run is supported by the previous notebook narrative rather than saved per-question outputs. The committed and verification runs have recorded evaluation outputs.

### Limitations

The evaluation contains only 15 answerable questions and 5 out-of-corpus questions.

Although temperature was set to 0, language-model responses may vary between runs.

Citation-ID groundedness measures whether cited chunk IDs were retrieved. It does not independently verify that every statement is semantically supported or clinically correct.

The 5/5 post-reranking refusals were verified in the current run only. They do not guarantee refusal reliability on unseen questions.

**Decision:** Retain cross-encoder re-ranking. The M3 groundedness target was met in the three reported runs, with the strongest direct evidence coming from the committed and verification runs. The verification run additionally confirms 5/5 refusals after re-ranking.


In [29]:
# Cell 13 - Consolidated M3 results: three runs

# Verify the new refusal test was actually executed
assert refused_after is not None, (
    "Run Cell 12b before saving results"
)
assert len(refusal_checks_after) == 5
assert len(out_of_scope) == 5

# Three reported evaluation runs
summary = pd.DataFrame([
    {
        "run": "early (reported)",
        "groundedness_before": 0.600,
        "groundedness_after": 0.9333333333333333,
        "wrong_refusals_before": 6,
        "refusals_before": "Not separately documented",
        "refusals_after": "Not evaluated",
        "evidence": "Earlier notebook narrative only"
    },
    {
        "run": "committed (2026-10-30)",
        "groundedness_before": 0.7333333333333333,
        "groundedness_after": 0.9333333333333333,
        "wrong_refusals_before": 4,
        "refusals_before": "5/5",
        "refusals_after": "Not evaluated",
        "evidence": "Original notebook and m3_answers.csv"
    },
    {
        "run": "verification (2026-10-31)",
        "groundedness_before": before.groundedness.mean(),
        "groundedness_after": after.groundedness.mean(),
        "wrong_refusals_before": int(before.refused.sum()),
        "refusals_before": f"{refused}/5",
        "refusals_after": f"{refused_after}/5",
        "evidence": "New Colab evaluation"
    }
])

summary["improvement"] = (
    summary["groundedness_after"]
    - summary["groundedness_before"]
).round(3)

# Save review-only files
summary_path = os.path.join(
    AGENT_OUT, "m3_summary_colab_review.csv"
)

answers_path = os.path.join(
    AGENT_OUT, "m3_answers_colab_review.csv"
)

refusals_path = os.path.join(
    AGENT_OUT, "m3_refusals_after_colab_review.csv"
)

summary.to_csv(summary_path, index=False)

# Per-question answers from the NEW run only
pd.concat([
    before.assign(run="before"),
    after.assign(run="after (rerank)")
], ignore_index=True).to_csv(
    answers_path, index=False
)

# Refusal details from the NEW reranked test
pd.DataFrame(refusal_checks_after).to_csv(
    refusals_path, index=False
)

print("M3 - THREE REPORTED RUNS")
print(summary.to_string(index=False))


print("\nNumber support verification:")
print(f"Before: {int(before['numbers_ok'].sum())}/{len(before)}")
print(f"After:  {int(after['numbers_ok'].sum())}/{len(after)}")


print("\nReview files saved:")
print(summary_path)
print(answers_path)
print(refusals_path)

print("\nOriginal GitHub reports were NOT overwritten.")



M3 - THREE REPORTED RUNS
                      run  groundedness_before  groundedness_after  wrong_refusals_before           refusals_before refusals_after                             evidence  improvement
         early (reported)             0.600000            0.933333                      6 Not separately documented  Not evaluated      Earlier notebook narrative only        0.333
   committed (2026-10-30)             0.733333            0.933333                      4                       5/5  Not evaluated Original notebook and m3_answers.csv        0.200
verification (2026-10-31)             0.733333            0.933333                      4                       5/5            5/5                 New Colab evaluation        0.200

Number support verification:
Before: 15/15
After:  15/15

Review files saved:
/content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/reports/agent/m3_summary_colab_review.csv
/content/drive/MyDrive/לימודים/Data Scienc

## Summary

### Corpus
| Check | Expected | Actual |
|---|---|---|
| NCI PDQ documents | 7 | **7** ✓ |
| Sections / chunks | — | 224 sections · **724 chunks** |


### M3

| Check | Expected | Actual |
|---|---|---|
| Frozen questions | 20 (15 + 5) | **20** ✓ |
| Refusals before re-ranking | 5 / 5 | **5 / 5** in committed and verification runs; early run not separately documented |
| Refusals after re-ranking | 5 / 5 | **5 / 5** in verification run; not evaluated in earlier runs |
| Groundedness, before | — | **0.600** early (reported), **0.733** committed, **0.733** verification |
| Groundedness, after re-ranking | ≥ 0.9 | **0.933** in all three reported runs ✓ |
| Improvement | — | **+0.333** early (reported), **+0.200** committed, **+0.200** verification |
| Numbers supported by cited chunks (verification run) | All | **15 / 15** before and after ✓ |


### Decisions (Module 5 template: what, why, the rejected alternative, evidence)
| Decision | Choice | Why | Rejected alternative | Evidence |
|---|---|---|---|---|
| Corpus | 7 NCI PDQ Health Professional summaries | authoritative, free to reuse as text, written for clinicians | general web search | frozen in `nci_corpus.json` |
| Chunking | ~900 chars, 150 overlap, sentence boundaries | PDQ sections are long; one idea per chunk | whole sections | 724 chunks from 224 sections |
| Evaluation | 15 answerable + 5 out-of-corpus questions, frozen before evaluation | measures citation-ID groundedness, number support, and refusal behavior | answerable questions only | Committed run: 5/5 refusals before re-ranking. Verification run: 5/5 before and after re-ranking. |
| Improvement | cross-encoder re-ranking (retrieve 20 → keep 3) | embedding-only retrieval sometimes missed relevant passages | larger k without re-ranking | Before: 0.600 (early, reported), 0.733 (committed), 0.733 (verification). After: 0.933 in all three reported runs. Gains: +0.333, +0.200, +0.200. |


### What the agent uses from this notebook
- The frozen corpus and the same chunk ids: every citation in the dermatologist's report is one of these ids.
- Re-ranking, kept here, is switched on in the agent's retrieval tool.
- Adults (18+) are never shown the two childhood pages; the query is built by code from the symptoms and body site.

**Next:** `08_agent_m4_vs_ifelse.ipynb` — is the agent needed at all?